In [1]:
from stable_platform_matchings import Optimizer, InstanceGenerator, OptimizerParams, SolverOptions

In [ ]:
SIM_SIZE = 12
N_INTS = 12
SEED = 67

INDO_CRS = "EPSG:23867"
DATA_DIR = "../../SyntheticInstanceGenerator/data/"

FARMERS_PATH = DATA_DIR + "farmers.csv"
FARMERS_14_PATH = DATA_DIR + "farmers_14.csv"
INTS_PATH = DATA_DIR + "intermediaries.csv"
GRAPH_PATH = DATA_DIR + "graph_0-14960_00_new.pickle"
ALPHA_PATH = DATA_DIR + "precomputed_alpha.json"
SIGMAS_PATH = DATA_DIR + "precomputed_sigmas.json"

In [3]:
ig = InstanceGenerator(
    FARMERS_PATH,
    FARMERS_14_PATH,
    INTS_PATH,
    GRAPH_PATH,
    ALPHA_PATH,
    SIGMAS_PATH
)

In [4]:
ig.gen_intermediaries(n_intermediaries=12, seed=SEED)
ig.gen_pickups(seed=SEED, scale=1, n_cycles=10)

In [5]:
platform = ig.gen_instance(
    instance_id=100,
    day=100,
    n_hist_sets=3,
    dev_mode="required_only"
)

In [6]:
ig.pickups_df

,farmer_id,farmer_x,farmer_y,farmer_lon,farmer_lat,cycle_phase,quantity,intermediary_id,nominal_day,schedule_offset,day,scaled_quantity
2,nostalgic_bohr_d0_f0,885117.880185,-77432.515101,102.459096,-0.699271,0,1.1,nostalgic_bohr,14,5,19,1.1
3,nostalgic_bohr_d0_f0,885117.880185,-77432.515101,102.459096,-0.699271,0,1.1,nostalgic_bohr,28,-4,24,1.1
5,nostalgic_bohr_d0_f0,885117.880185,-77432.515101,102.459096,-0.699271,0,1.1,nostalgic_bohr,56,1,57,1.1
6,nostalgic_bohr_d0_f0,885117.880185,-77432.515101,102.459096,-0.699271,0,1.1,nostalgic_bohr,70,0,70,1.1
7,nostalgic_bohr_d0_f0,885117.880185,-77432.515101,102.459096,-0.699271,0,1.1,nostalgic_bohr,84,-3,81,1.1
...,...,...,...,...,...,...,...,...,...,...,...,...
3594,practical_ishizaka_d13_f0,879867.880185,-36932.515101,102.411803,-0.333544,13,0.6,practical_ishizaka,83,0,83,0.6
3595,practical_ishizaka_d13_f0,879867.880185,-36932.515101,102.411803,-0.333544,13,0.6,practical_ishizaka,97,0,97,0.6
3596,practical_ishizaka_d13_f0,879867.880185,-36932.515101,102.411803,-0.333544,13,0.6,practical_ishizaka,111,-5,106,0.6
3597,practical_ishizaka_d13_f0,879867.880185,-36932.515101,102.411803,-0.333544,13,0.6,practical_ishizaka,125,0,125,0.6


In [7]:
epsilons = {intermediary.id: 2 for intermediary in platform.intermediaries}
het_costs = {intermediary.id: (platform.dist_to_mill[intermediary.id] * 2) for intermediary in platform.intermediaries}

In [8]:
params = OptimizerParams(
    het_costs=het_costs,
    epsilons=epsilons,
    backend="gurobi",
    vrp_mode="approximate",
    verbose=True,
    print_width=80
)

opt = Optimizer(platform, params)



============================= Optimizer Parameters =============================
---------------------------------- het_costs -----------------------------------
  {'flamboyant_keldysh': 541213.5021463943,
   'happy_poitras': 634555.8414770004,
   'hardcore_edison': 408198.89799107745,
   'keen_benz': 543952.1088963698,
   'nervous_roentgen': 454001.0212905022,
   'nostalgic_bohr': 229550.8889446209,
   'nostalgic_dijkstra': 480596.4099848486,
   'practical_ishizaka': 549172.8694691922,
   'suspicious_chaum': 580528.1411912779,
   'suspicious_joliot': 291725.66334128054,
   'vibrant_lumiere': 530571.677306222,
   'wizardly_elgamal': 593049.7210509629}
----------------------------------- epsilons -----------------------------------
  {'flamboyant_keldysh': 2,
   'happy_poitras': 2,
   'hardcore_edison': 2,
   'keen_benz': 2,
   'nervous_roentgen': 2,
   'nostalgic_bohr': 2,
   'nostalgic_dijkstra': 2,
   'practical_ishizaka': 2,
   'suspicious_chaum': 2,
   'suspicious_joliot': 2,
   

In [9]:
options = SolverOptions(
    strategy="heuristic_optimized",
    structured_farmer_payments=False,
    dominance_constraints=False,
    pay_unmatched=False,
    aggregate=True
)


solution = opt.solve(options=options)

if solution.max_intermediary_welfare_solution is not None:
    print(solution.max_intermediary_welfare_solution.platform_profit/platform.lc_to_usd)



================================ Solver Options ================================
  Strategy                   heuristic_optimized
  Structured Farmer Payments False
  Dominance Constraints      False
  Early Stop                 False
  Aggregate                  True
  Pay Unmatched              False


======================== Strategy: heuristic_optimized =========================
  Farmers                    12
  Intermediaries             12


============================== Branch Evaluation ===============================
  Forced matched:
    []
  Forced unmatched:
    []

Set parameter Threads to value 0
----------------------- Primal Solve: Forced Lower Bound -----------------------
  Platform-profit objective  974,066.715
  Intermediary-welfare objective 2,567,011.733
  Farmer-welfare objective   41,677,088.680
  New rows                   152
  Intermediary probabilities:
    {'flamboyant_keldysh': 0.0,
     'happy_poitras': 0.0,
     'hardcore_edison': 1.0,
     'keen_ben

In [12]:
summary = opt.solve(
    "heuristic_optimized", 
    options={
        "structured_farmer_payments": False,
        "domination": False,
        "pay_unmatched": True,
        "aggregate": True
    }
)

if summary.max_intermediary_welfare_solution is not None:
    print(summary.max_intermediary_welfare_solution.platform_profit/platform.lc_to_usd)



============================= Dominance Relations ==============================
  Number of relations        41
  Relations:
    [('stoic_pasteur', 'goofy_kalam'),
     ('infallible_morse', 'goofy_kalam'),
     ('optimistic_lalande', 'goofy_kalam'),
     ('eloquent_hertz', 'goofy_kalam'),
     ('stoic_ramanujan', 'goofy_kalam'),
     ('silly_jemison', 'goofy_kalam'),
     ('frosty_cerf', 'goofy_kalam'),
     ('goofy_kalam', 'wonderful_jepsen'),
     ('stoic_pasteur', 'awesome_hopper'),
     ('stoic_pasteur', 'infallible_morse'),
     ('stoic_pasteur', 'optimistic_lalande'),
     ('stoic_pasteur', 'relaxed_turing'),
     ('stoic_pasteur', 'stoic_ramanujan'),
     ('stoic_pasteur', 'affectionate_grothendieck'),
     ('stoic_pasteur', 'frosty_cerf'),
     ('stoic_pasteur', 'wonderful_jepsen'),
     ('optimistic_lalande', 'awesome_hopper'),
     ('awesome_hopper', 'relaxed_turing'),
     ('frosty_cerf', 'awesome_hopper'),
     ('awesome_hopper', 'wonderful_jepsen'),
     ('optimistic_la

In [12]:
summary = opt.solve(
    "heuristic_optimized", 
    options={
        "structured_farmer_payments": True,
        "domination": False,
        "pay_unmatched": False,
        "aggregate": True
    }
)

if summary.max_intermediary_welfare_solution is not None:
    print(summary.max_intermediary_welfare_solution.platform_profit/platform.lc_to_usd)



============================= Dominance Relations ==============================
  Number of relations        41
  Relations:
    [('stoic_pasteur', 'goofy_kalam'),
     ('infallible_morse', 'goofy_kalam'),
     ('optimistic_lalande', 'goofy_kalam'),
     ('eloquent_hertz', 'goofy_kalam'),
     ('stoic_ramanujan', 'goofy_kalam'),
     ('silly_jemison', 'goofy_kalam'),
     ('frosty_cerf', 'goofy_kalam'),
     ('goofy_kalam', 'wonderful_jepsen'),
     ('stoic_pasteur', 'awesome_hopper'),
     ('stoic_pasteur', 'infallible_morse'),
     ('stoic_pasteur', 'optimistic_lalande'),
     ('stoic_pasteur', 'relaxed_turing'),
     ('stoic_pasteur', 'stoic_ramanujan'),
     ('stoic_pasteur', 'affectionate_grothendieck'),
     ('stoic_pasteur', 'frosty_cerf'),
     ('stoic_pasteur', 'wonderful_jepsen'),
     ('optimistic_lalande', 'awesome_hopper'),
     ('awesome_hopper', 'relaxed_turing'),
     ('frosty_cerf', 'awesome_hopper'),
     ('awesome_hopper', 'wonderful_jepsen'),
     ('optimistic_la

In [21]:
summary = opt.solve(
    "heuristic_optimized", 
    options={
        "structured_farmer_payments": False,
        "domination": True,
        "pay_unmatched": False,
        "aggregate": True
    }
)

if summary.max_intermediary_welfare_solution is not None:
    print(summary.max_intermediary_welfare_solution.platform_profit/platform.lc_to_usd)


============================= Dominance Relations ==============================
  Number of relations        35
  Relations:
    [('stoic_pasteur', 'goofy_kalam'),
     ('infallible_morse', 'goofy_kalam'),
     ('optimistic_lalande', 'goofy_kalam'),
     ('stoic_ramanujan', 'goofy_kalam'),
     ('goofy_kalam', 'affectionate_grothendieck'),
     ('silly_jemison', 'goofy_kalam'),
     ('frosty_cerf', 'goofy_kalam'),
     ('stoic_pasteur', 'infallible_morse'),
     ('stoic_pasteur', 'affectionate_grothendieck'),
     ('silly_jemison', 'stoic_pasteur'),
     ('frosty_cerf', 'stoic_pasteur'),
     ('awesome_hopper', 'relaxed_turing'),
     ('awesome_hopper', 'affectionate_grothendieck'),
     ('awesome_hopper', 'wonderful_jepsen'),
     ('optimistic_lalande', 'infallible_morse'),
     ('stoic_ramanujan', 'infallible_morse'),
     ('infallible_morse', 'affectionate_grothendieck'),
     ('silly_jemison', 'infallible_morse'),
     ('frosty_cerf', 'infallible_morse'),
     ('stoic_ramanujan',